Dataframe creation

In [ ]:
import pandas as pd

In [ ]:

mutation= pd.read_csv("/content/HS_CPTAC_LUAD_somatic_mutation_gene.cbt", sep="\t")


In [ ]:
scnv_tumor_log = pd.read_csv("/content/HS_CPTAC_LUAD_cnv_gene_LR_scnvtumor.cct", sep="\t")


In [ ]:
methylation_normal = pd.read_csv("/content/HS_CPTAC_LUAD_methylation_mean_gene_normal.cct", sep="\t")

In [ ]:
methylation_tumor = pd.read_csv("/content/HS_CPTAC_LUAD_methylation_mean_gene_tumor.cct", sep="\t")

In [ ]:
rna_tumor=pd.read_csv("/content/HS_CPTAC_LUAD_rnaseq_uq_rpkm_log2_NArm_TUMOR.cct", sep="\t")

FileNotFoundError: [Errno 2] No such file or directory: '/content/HS_CPTAC_LUAD_rnaseq_uq_rpkm_log2_NArm_TUMOR.cct'

In [ ]:
rna_normal=pd.read_csv("/content/HS_CPTAC_LUAD_rnaseq_uq_rpkm_log2_NArm_NORMAL.cct", sep="\t")

In [ ]:
proteome_tumor=pd.read_csv("/content/HS_CPTAC_LUAD_proteome_ratio_NArm_TUMOR.cct", sep="\t")

In [ ]:
proteome_normal=pd.read_csv("/content/HS_CPTAC_LUAD_proteome_ratio_NArm_NORMAL.cct", sep="\t")

In [ ]:
mutation.shape

In [ ]:
scnv_tumor_log.shape

In [ ]:
methylation_tumor.shape

In [ ]:
methylation_normal.shape

In [ ]:
rna_tumor.shape

In [ ]:
rna_normal.shape

In [ ]:
proteome_normal.shape

In [ ]:
proteome_tumor.shape

Preprocessing all df are already clean

In [ ]:
# Get common sample columns across ALL dfs

common_cols = (
    set(methylation_normal.columns)
    & set(mutation.columns)
    & set(scnv_tumor_log.columns)
    & set(methylation_tumor.columns)
    & set(rna_tumor.columns)
    & set(rna_normal.columns)
    & set(proteome_tumor.columns)
    & set(proteome_normal.columns)
)

# remove non-sample column
common_cols.discard("GeneSymbol")

# optional: keep consistent order
common_cols = sorted(list(common_cols))

In [ ]:
len(common_cols)

In [ ]:
# subset all dfs
methylation_normal = methylation_normal[
    ["GeneSymbol"] + common_cols
]

mutation = mutation[
    ["GeneSymbol"] + common_cols
]



scnv_tumor_log = scnv_tumor_log[
    ["GeneSymbol"] + common_cols
]

methylation_tumor = methylation_tumor[
    ["GeneSymbol"] + common_cols
]

rna_tumor = rna_tumor[
    ["GeneSymbol"] + common_cols
]

rna_normal = rna_normal[
    ["GeneSymbol"] + common_cols
]

proteome_tumor = proteome_tumor[
    ["GeneSymbol"] + common_cols
]

proteome_normal = proteome_normal[
    ["GeneSymbol"] + common_cols
]


In [ ]:


# Verify shapes
print("methylation_normal:", methylation_normal.shape)
print("mutation:", mutation.shape)

print("scnv_tumor_log:", scnv_tumor_log.shape)
print("methylation_tumor:", methylation_tumor.shape)
print("rna_tumor:", rna_tumor.shape)
print("rna_normal:", rna_normal.shape)
print("proteome_tumor:", proteome_tumor.shape)
print("proteome_normal:", proteome_normal.shape)



In [ ]:
print("methylation_normal:\n", methylation_normal.isnull().sum())
print("mutation:\n", mutation.isnull().sum())

print("scnv_tumor_log:\n", scnv_tumor_log.isnull().sum())
print("methylation_tumor:\n", methylation_tumor.isnull().sum())
print("rna_tumor:\n", rna_tumor.isnull().sum())
print("rna_normal:\n", rna_normal.isnull().sum())
print("proteome_tumor:\n", proteome_tumor.isnull().sum())
print("proteome_normal:\n", proteome_normal.isnull().sum())

In [ ]:
# Verify identical columns
print("\nColumn alignment checks:")

print((mutation.columns == methylation_normal.columns).all())

print((scnv_tumor_log.columns == methylation_normal.columns).all())
print((methylation_tumor.columns == methylation_normal.columns).all())
print((rna_tumor.columns == methylation_normal.columns).all())
print((rna_normal.columns == methylation_normal.columns).all())
print((proteome_tumor.columns == methylation_normal.columns).all())
print((proteome_normal.columns == methylation_normal.columns).all())

In [ ]:
# Save RNA datasets
rna_tumor.to_csv("rna_tumor_filtered.csv", index=False)
rna_normal.to_csv("rna_normal_filtered.csv", index=False)

# Save proteome datasets
proteome_tumor.to_csv("proteome_tumor_filtered.csv", index=False)
proteome_normal.to_csv("proteome_normal_filtered.csv", index=False)

print("RNA and proteome files saved successfully.")

In [ ]:
# Verify same sample columns
print("\nColumn alignment check:")

print((mutation.columns == methylation_normal.columns).all())

print((scnv_tumor_log.columns == methylation_normal.columns).all())
print((methylation_tumor.columns == methylation_normal.columns).all())

Mutation

In [ ]:
sample_cols = mutation.columns[1:]

In [ ]:
len(sample_cols)

In [ ]:
# convert mutation values to numeric
mutation[sample_cols] = mutation[sample_cols].apply(
    pd.to_numeric,
    errors='coerce'
)

In [ ]:
sample_cols

In [ ]:
mutation

In [ ]:
mutation.isnull().sum().sum()

In [ ]:
# ensure binary values
mutation[sample_cols] = (mutation[sample_cols] > 0).astype(int)

In [ ]:
# number of mutated samples per gene
mutation["Frequency"] = mutation[sample_cols].sum(axis=1)

# mutation percentage
mutation["Percent"] = (
    mutation["Frequency"] / len(sample_cols)
) * 100

top_mutated = mutation.sort_values(
    by="Frequency",
    ascending=False
)

In [ ]:
top10__mutated = mutation.sort_values(
    by="Frequency",
    ascending=False
).head(10)

In [ ]:
top10__mutated

In [ ]:
top_mutated.head()

In [ ]:

# -----------------------------
# OPTIONAL:
# KEEP GENES MUTATED IN >=5%
# -----------------------------

threshold = 0.05 * len(sample_cols)

recurrent_genes = top_mutated[
    top_mutated["Frequency"] >= threshold
]

print(recurrent_genes[
    ["GeneSymbol", "Frequency", "Percent"]
].head())


In [ ]:
print("methylation_normal:", methylation_normal.isnull().sum().sum())
print("mutation:", mutation.isnull().sum().sum())

print("scnv_tumor_log:", scnv_tumor_log.isnull().sum().sum())
print("methylation_tumor:", methylation_tumor.isnull().sum().sum())


In [ ]:

 (methylation_normal.isnull().sum().sum() / methylation_normal.size) * 100

In [ ]:
 (methylation_tumor.isnull().sum().sum() / methylation_tumor.size) * 100

methylation

In [ ]:
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

In [ ]:
methylation_tumor

In [ ]:
methylation_normal

In [ ]:
# overall minimum value
print(methylation_tumor.iloc[:,1:].min().min())

# overall maximum value
print(methylation_tumor.iloc[:,1:].max().max())

In [ ]:
import numpy as np

In [ ]:
# overall minimum value
print(methylation_normal.iloc[:,1:].min().min())

# overall maximum value
print(methylation_normal.iloc[:,1:].max().max())

In [ ]:
methylation_tumor.iloc[:, 1:] = (
    methylation_tumor.iloc[:, 1:]
    .apply(pd.to_numeric, errors='coerce')
)

methylation_normal.iloc[:, 1:] = (
    methylation_normal.iloc[:, 1:]
    .apply(pd.to_numeric, errors='coerce')
)

In [ ]:
methylation_tumor['TumorMean'] = (
    methylation_tumor.iloc[:, 1:]
    .mean(axis=1)
)

methylation_normal['NormalMean'] = (
    methylation_normal.iloc[:, 1:]
    .mean(axis=1)
)

In [ ]:
# Merge properly by GeneSymbol

methyl_difference = pd.merge(
    methylation_tumor[['GeneSymbol', 'TumorMean']],
    methylation_normal[['GeneSymbol', 'NormalMean']],
    on='GeneSymbol',
    how='inner'
)


methyl_difference['DeltaMethylation'] = (
    methyl_difference['TumorMean']
    - methyl_difference['NormalMean']
)

In [ ]:

methyl_difference

In [ ]:
def methylation_type(delta, threshold=0.1):

    if delta > threshold:
        return "Hypermethylated"

    elif delta < -threshold:
        return "Hypomethylated"

    else:
        return "NoChange"



In [ ]:
methyl_difference['MethylationType'] = (
    methyl_difference['DeltaMethylation']
    .apply(methylation_type)
)

In [ ]:
methyl_difference['MethylationType'].value_counts()

In [ ]:
tumor_df = (
    methylation_tumor
    .drop(columns='TumorMean')
    .set_index('GeneSymbol')
)

normal_df = (
    methylation_normal
    .drop(columns='NormalMean')
    .set_index('GeneSymbol')
)

common_genes = (
    tumor_df.index
    .intersection(normal_df.index)
)

In [ ]:
tumor_df

In [ ]:
common_genes

In [ ]:
meth_results = []

for gene in common_genes:

    tumor_vals = tumor_df.loc[gene].dropna()
    normal_vals = normal_df.loc[gene].dropna()

    # skip genes with too few samples
    if len(tumor_vals) < 3 or len(normal_vals) < 3:
        continue

    t_stat, p_value = ttest_ind(
        tumor_vals,
        normal_vals,
        equal_var=False
    )

    meth_results.append([
        gene,
        t_stat,
        p_value
    ])

In [ ]:
meth_pvals = pd.DataFrame(
    meth_results,
    columns=[
        'GeneSymbol',
        't_statistic',
        'p_value'
    ]
)

In [ ]:
meth_pvals['adj_p_value'] = multipletests(
    meth_pvals['p_value'],
    method='fdr_bh'
)[1]

In [ ]:
meth_pvals

In [ ]:
methyl_difference = pd.merge(
    methyl_difference,
    meth_pvals,
    on='GeneSymbol',
    how='left'
)

In [ ]:
methyl_difference

In [ ]:
methyl_difference

In [ ]:
lower = np.percentile( methyl_difference["DeltaMethylation"],5 )
upper = np.percentile( methyl_difference["DeltaMethylation"], 95)

In [ ]:
lower

In [ ]:
upper

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(
    methyl_difference["DeltaMethylation"].dropna(),
    bins=50
)

plt.axvline(0.1, color='red', linestyle='--')
plt.axvline(-0.1, color='red', linestyle='--')

plt.xlabel("Delta Beta (Tumor - Normal)")
plt.ylabel("Gene Count")
plt.title("Distribution of Methylation Difference (Δβ)")

plt.show()

In [ ]:


significant_methylation = methyl_difference[
    (methyl_difference['adj_p_value'] < 0.05) &
    (abs(methyl_difference['DeltaMethylation']) > 0.1)
]

# =========================
# 13. Top hypermethylated genes
# =========================

top_hyper = (
    significant_methylation[
        significant_methylation['MethylationType']
        == 'Hypermethylated'
    ]
    .sort_values(
        by=[
            'adj_p_value',
            'DeltaMethylation'
        ],
        ascending=[True, False]
    )

)


top_hypo = (
    significant_methylation[
        significant_methylation['MethylationType']
        == 'Hypomethylated'
    ]
    .sort_values(
        by=[
            'adj_p_value',
            'DeltaMethylation'
        ],
        ascending=[True, True]
    )

)



In [ ]:
top_hypo

In [ ]:
top_hyper

SCNV

In [ ]:
# numeric conversion
scnv_tumor_log.iloc[:, 1:] = scnv_tumor_log.iloc[:, 1:].apply(
    pd.to_numeric,
    errors='coerce'
)

# remove duplicate genes
tumor_df = scnv_tumor_log.set_index("GeneSymbol")
tumor_df = tumor_df.groupby(level=0).mean()

# calculate mean CNV
mean_cnv = tumor_df.mean(axis=1).reset_index()
mean_cnv.columns = ["GeneSymbol", "MeanCNV"]

In [ ]:
tumor_df

In [ ]:
mean_cnv

In [ ]:
mean_cnv

In [ ]:


# classify CNV
threshold = 0.2

def cnv_type(val):

    if val > threshold:
        return "Amplified"

    elif val < -threshold:
        return "Deleted"

    else:
        return "NoChange"

mean_cnv["CNV_type"] = mean_cnv["MeanCNV"].apply(cnv_type)

# top amplified
top_amplified = mean_cnv[
    mean_cnv["CNV_type"] == "Amplified"
].sort_values(
    by="MeanCNV",
    ascending=False
)

# top deleted
top_deleted = mean_cnv[
    mean_cnv["CNV_type"] == "Deleted"
].sort_values(
    by="MeanCNV",
    ascending=True
)



In [ ]:
import numpy as np

In [ ]:
lower = np.percentile(mean_cnv["MeanCNV"],5 )
upper = np.percentile(mean_cnv["MeanCNV"], 95)

In [ ]:
lower

In [ ]:
upper

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(
    mean_cnv["MeanCNV"].dropna(),
    bins=50
)

plt.axvline(0.15, color='red', linestyle='--')
plt.axvline(-0.15, color='red', linestyle='--')

plt.xlabel("Mean CNV")
plt.ylabel("Gene Count")
plt.title("Distribution of Mean Copy Number Variation")

plt.show()

In [ ]:
top_amplified

In [ ]:
top_deleted

In [ ]:
top10__mutated

In [ ]:
top_10_hyper

In [ ]:
top_10_hypo

In [ ]:
top_10_hyper = (
    significant_methylation[
        significant_methylation['MethylationType']
        == 'Hypermethylated'
    ]
    .sort_values(
        by=[
            'adj_p_value',
            'DeltaMethylation'
        ],
        ascending=[True, False]
    )

).head(10)


top_10_hypo = (
    significant_methylation[
        significant_methylation['MethylationType']
        == 'Hypomethylated'
    ]
    .sort_values(
        by=[
            'adj_p_value',
            'DeltaMethylation'
        ],
        ascending=[True, True]
    )

).head(10)

In [ ]:
top_10_amplified

In [ ]:
# top amplified
top_10_amplified = mean_cnv[
    mean_cnv["CNV_type"] == "Amplified"
].sort_values(
    by="MeanCNV",
    ascending=False
).head(10)

# top deleted
top_10_deleted = mean_cnv[
    mean_cnv["CNV_type"] == "Deleted"
].sort_values(
    by="MeanCNV",
    ascending=True
).head(10)



In [ ]:
top_10_deleted

In [ ]:
mut_genes = top10__mutated["GeneSymbol"].tolist()
hyper_genes = top_10_hyper["GeneSymbol"].tolist()
hypo_genes = top_10_hypo["GeneSymbol"].tolist()
amp_genes = top_10_amplified["GeneSymbol"].tolist()
del_genes = top_10_deleted["GeneSymbol"].tolist()

import pandas as pd

final_table = pd.DataFrame({
    "Top_Mutated": mut_genes,
    "Top_HyperMethylated": hyper_genes,
    "Top_HypoMethylated": hypo_genes,
    "Top_Amplified": amp_genes,
    "Top_Deleted": del_genes
})

final_table

In [ ]:
import pandas as pd

# ---------------------------
# Mutation (top 10)
# ---------------------------
mut = top_mutated[["GeneSymbol", "Frequency", "Percent"]].copy()
mut.rename(columns={
    "Frequency": "Mut_Frequency",
    "Percent": "Mut_Percent"
}, inplace=True)

# ---------------------------
# Hypermethylated (top 10)
# ---------------------------
hyper = top_hyper[["GeneSymbol", "DeltaMethylation"]].copy()
hyper.rename(columns={"DeltaMethylation": "Hyper_DeltaMeth"}, inplace=True)

# ---------------------------
# Hypomethylated (top 10)
# ---------------------------
hypo = top_hypo[["GeneSymbol", "DeltaMethylation"]].copy()
hypo.rename(columns={"DeltaMethylation": "Hypo_DeltaMeth"}, inplace=True)

# ---------------------------
# CNV Amplified (top 10)
# ---------------------------
amp = top_amplified[["GeneSymbol", "MeanCNV"]].copy()
amp.rename(columns={"MeanCNV": "CNV_Amplified"}, inplace=True)

dele = top_deleted[["GeneSymbol", "MeanCNV"]].copy()
dele.rename(columns={"MeanCNV": "CNV_deleted"}, inplace=True)

# ---------------------------
# Merge all datasets
# ---------------------------
b = mut.merge(hyper, on="GeneSymbol", how="outer") \
        .merge(hypo, on="GeneSymbol", how="outer") \
        .merge(amp, on="GeneSymbol", how="outer") \
        .merge(dele, on="GeneSymbol", how="outer")

b=b.fillna(0)
# ---------------------------
# Final view
# ---------------------------
b

In [ ]:
rna=pd.read_csv("/content/DE_results_filtered.csv")

In [ ]:
rna

In [ ]:
pro=pd.read_csv("/content/results.csv")

In [ ]:
pro

In [ ]:
merged[:]=0

In [ ]:
import pandas as pd


# ---------------------------
# 3. RNA-seq (log2FC already present)
# ---------------------------
rna = rna[['Gene', 'log2FoldChange']].copy()
rna.rename(columns={
    'Gene': 'GeneSymbol',
    'log2FoldChange': 'RNA_logFC'
}, inplace=True)

# ---------------------------
# 4. Proteomics (logFC already present)
# ---------------------------
pro = pro[['GeneSymbol', 'logFC']].copy()
pro.rename(columns={'logFC': 'PROT_logFC'}, inplace=True)

# ---------------------------
# 5. Merge all (Gene-based)
# ---------------------------
x = b.merge(rna, on='GeneSymbol', how='outer') \
               .merge(pro, on='GeneSymbol', how='outer')

# ---------------------------
# 6. Fill missing values
# ---------------------------
x = x.fillna(0)
x

In [226]:
x

In [227]:
df=x
df['gen_flag'] = (df['Mut_Frequency'] > 0) | (df['CNV_Amplified'] > 0) | (df['CNV_deleted'] > 0)

df['epi_flag'] = (df['Hyper_DeltaMeth'] > 0) | (df['Hypo_DeltaMeth'] > 0)

df['rna_flag'] = abs(df['RNA_logFC']) > 1

df['prot_flag'] = abs(df['PROT_logFC']) > 1

NameError: name 'x' is not defined

In [228]:
df

NameError: name 'df' is not defined

In [229]:
all_layers_genes = df[
    (df['gen_flag']) &
    (df['epi_flag']) &
    (df['rna_flag']) &
    (df['prot_flag'])
]

In [230]:
all_layers_genes

NameError: name 'all_layers_genes' is not defined

In [231]:
df['support_3'] = (
    df['gen_flag'].astype(int) +
    df['epi_flag'].astype(int) +
    df['rna_flag'].astype(int) +
    df['prot_flag'].astype(int)
)

genes_3_layer = df[df['support_3'] >= 3]

In [232]:
genes_3_layer